# SageMaker Pipeline: AutoGluon Multimodal (Text + Tabular)

End-to-end ML pipeline using SageMaker SDK v3:

1. **ProcessingStep** - Preprocess JSONL data into train/validation/test splits
2. **TrainingStep** - AutoGluon MultiModalPredictor training (GPU)
3. **ProcessingStep** - Evaluate model on test data (CPU — GPU processing quota may be limited)

## Configuration

In [ ]:
import os

import boto3
import sagemaker

REGION = boto3.session.Session().region_name
sess = sagemaker.session.Session()
BUCKET = sess.default_bucket()
S3_PREFIX = "autogluon-multimodal"
AG_VERSION = "1.5"
PY_VERSION = "py312"
PIPELINE_NAME = "AutoGluonMultimodalPipeline"

# Paths to container scripts (relative to this notebook)
preprocess_script = os.path.join("..", "0-data-prep", "preprocess.py")
train_source_dir = os.path.join("..", "1-training")
evaluate_script = "run_evaluation.py"

print(f"Region:   {REGION}")
print(f"Bucket:   {BUCKET}")
print(f"Pipeline: {PIPELINE_NAME}")

## Discover IAM Role

In [ ]:
iam = boto3.client("iam")
role_arn = None
paginator = iam.get_paginator("list_roles")
for page in paginator.paginate():
    for role in page["Roles"]:
        if "SageMaker" in role["RoleName"] or "sagemaker" in role["RoleName"]:
            role_arn = role["Arn"]
            break
    if role_arn:
        break

if not role_arn:
    raise ValueError("No SageMaker IAM role found. Set role_arn manually.")

print(f"Role: {role_arn}")

## Imports

In [ ]:
from sagemaker.core import image_uris
from sagemaker.core.processing import (
    ProcessingInput,
    ProcessingOutput,
    ScriptProcessor,
)
from sagemaker.core.shapes.shapes import ProcessingS3Input, ProcessingS3Output
from sagemaker.core.training.configs import (
    Compute,
    OutputDataConfig,
    SourceCode,
    StoppingCondition,
)
from sagemaker.core.workflow.parameters import ParameterFloat, ParameterString
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.core.workflow.properties import PropertyFile
from sagemaker.mlops.workflow.pipeline import Pipeline
from sagemaker.mlops.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.train import ModelTrainer

## Define `create_pipeline()`

In [ ]:
def create_pipeline(
    role_arn: str,
    region: str,
    bucket: str,
    ag_version: str = "1.5",
    py_version: str = "py312",
    pipeline_name: str = "AutoGluonMultimodalPipeline",
):
    pipeline_session = PipelineSession()
    s3_prefix = f"s3://{bucket}/autogluon-multimodal/pipeline"

    # -------------------------------------------------------------------------
    # Pipeline Parameters
    # -------------------------------------------------------------------------
    input_data_uri = ParameterString(
        name="InputDataUri",
        default_value=f"s3://{bucket}/autogluon-multimodal/raw/",
    )
    training_instance_type = ParameterString(
        name="TrainingInstanceType", default_value="ml.g4dn.xlarge"
    )
    roc_auc_threshold = ParameterFloat(name="RocAucThreshold", default_value=0.75)

    # -------------------------------------------------------------------------
    # Image URIs
    # -------------------------------------------------------------------------
    ag_training_image = image_uris.retrieve(
        "autogluon",
        region=region,
        version=ag_version,
        py_version=py_version,
        image_scope="training",
        instance_type="ml.g4dn.xlarge",
    )
    # CPU image for evaluation (GPU processing quota may be limited)
    ag_cpu_image = image_uris.retrieve(
        "autogluon",
        region=region,
        version=ag_version,
        py_version=py_version,
        image_scope="training",
        instance_type="ml.m5.xlarge",
    )
    processing_image = image_uris.retrieve("sklearn", region=region, version="1.2-1")

    # -------------------------------------------------------------------------
    # Step 1: Preprocessing (v3 step_args pattern)
    # -------------------------------------------------------------------------
    preprocessor = ScriptProcessor(
        image_uri=processing_image,
        role=role_arn,
        command=["python3"],
        instance_type="ml.m5.xlarge",
        instance_count=1,
        sagemaker_session=pipeline_session,
    )

    step_preprocess = ProcessingStep(
        name="PreprocessMultimodal",
        step_args=preprocessor.run(
            code=os.path.abspath(preprocess_script),
            inputs=[
                ProcessingInput(
                    input_name="input",
                    s3_input=ProcessingS3Input(
                        s3_uri=input_data_uri,
                        local_path="/opt/ml/processing/input",
                        s3_data_type="S3Prefix",
                    ),
                ),
            ],
            outputs=[
                ProcessingOutput(
                    output_name="train",
                    s3_output=ProcessingS3Output(
                        s3_uri=f"{s3_prefix}/processed/train/",
                        local_path="/opt/ml/processing/train",
                        s3_upload_mode="EndOfJob",
                    ),
                ),
                ProcessingOutput(
                    output_name="validation",
                    s3_output=ProcessingS3Output(
                        s3_uri=f"{s3_prefix}/processed/validation/",
                        local_path="/opt/ml/processing/validation",
                        s3_upload_mode="EndOfJob",
                    ),
                ),
                ProcessingOutput(
                    output_name="test",
                    s3_output=ProcessingS3Output(
                        s3_uri=f"{s3_prefix}/processed/test/",
                        local_path="/opt/ml/processing/test",
                        s3_upload_mode="EndOfJob",
                    ),
                ),
            ],
        ),
    )

    # -------------------------------------------------------------------------
    # Step 2: Training (GPU)
    # -------------------------------------------------------------------------
    trainer = ModelTrainer(
        training_image=ag_training_image,
        role=role_arn,
        source_code=SourceCode(
            source_dir=os.path.abspath(train_source_dir),
            entry_script="train.py",
        ),
        compute=Compute(
            instance_type=training_instance_type,
            instance_count=1,
            volume_size_in_gb=50,
            keep_alive_period_in_seconds=0,
        ),
        output_data_config=OutputDataConfig(
            s3_output_path=f"{s3_prefix}/model/",
        ),
        hyperparameters={
            "numerical-feature-names": "CustServ Calls,Account Length",
            "categorical-feature-names": "plan,limit",
            "textual-feature-names": "text",
            "label-name": "y",
            "problem_type": "classification",
            "eval_metric": "roc_auc",
            "presets": "medium_quality",
            "pretrained-transformer": "google/electra-small-discriminator",
            "verbosity": 2,
        },
        base_job_name="ag-multimodal-train",
        stopping_condition=StoppingCondition(max_runtime_in_seconds=14400),
        sagemaker_session=pipeline_session,
    )

    step_train = TrainingStep(
        name="TrainMultimodal",
        step_args=trainer.train(
            input_data_config=[
                {
                    "channel_name": "train",
                    "data_source": {
                        "s3_data_source": {
                            "s3_uri": step_preprocess.properties.ProcessingOutputConfig.Outputs[
                                "train"
                            ].S3Output.S3Uri,
                            "s3_data_type": "S3Prefix",
                        }
                    },
                },
                {
                    "channel_name": "validation",
                    "data_source": {
                        "s3_data_source": {
                            "s3_uri": step_preprocess.properties.ProcessingOutputConfig.Outputs[
                                "validation"
                            ].S3Output.S3Uri,
                            "s3_data_type": "S3Prefix",
                        }
                    },
                },
            ],
        ),
    )

    # -------------------------------------------------------------------------
    # Step 3: Evaluation (CPU — GPU processing job quota may be 0)
    # -------------------------------------------------------------------------
    evaluation_report = PropertyFile(
        name="EvaluationReport",
        output_name="evaluation",
        path="evaluation.json",
    )

    evaluator = ScriptProcessor(
        image_uri=ag_cpu_image,
        role=role_arn,
        command=["python3"],
        instance_type="ml.m5.xlarge",
        instance_count=1,
        sagemaker_session=pipeline_session,
    )

    step_evaluate = ProcessingStep(
        name="EvaluateMultimodal",
        step_args=evaluator.run(
            code=os.path.abspath(evaluate_script),
            inputs=[
                ProcessingInput(
                    input_name="model",
                    s3_input=ProcessingS3Input(
                        s3_uri=step_train.properties.ModelArtifacts.S3ModelArtifacts,
                        local_path="/opt/ml/processing/model",
                        s3_data_type="S3Prefix",
                    ),
                ),
                ProcessingInput(
                    input_name="test",
                    s3_input=ProcessingS3Input(
                        s3_uri=step_preprocess.properties.ProcessingOutputConfig.Outputs[
                            "test"
                        ].S3Output.S3Uri,
                        local_path="/opt/ml/processing/test",
                        s3_data_type="S3Prefix",
                    ),
                ),
            ],
            outputs=[
                ProcessingOutput(
                    output_name="evaluation",
                    s3_output=ProcessingS3Output(
                        s3_uri=f"{s3_prefix}/evaluation/",
                        local_path="/opt/ml/processing/evaluation",
                        s3_upload_mode="EndOfJob",
                    ),
                ),
            ],
        ),
        property_files=[evaluation_report],
    )

    # Note: Model registration (ConditionStep + ModelStep) removed —
    # v3 Model.register() is incompatible with PipelineSession/PipelineVariable.

    # -------------------------------------------------------------------------
    # Assemble Pipeline
    # -------------------------------------------------------------------------
    pipeline = Pipeline(
        name=pipeline_name,
        parameters=[
            input_data_uri,
            training_instance_type,
            roc_auc_threshold,
        ],
        steps=[step_preprocess, step_train, step_evaluate],
        sagemaker_session=pipeline_session,
    )

    return pipeline

## Create/Update and Execute Pipeline

In [ ]:
pipeline = create_pipeline(
    role_arn=role_arn,
    region=REGION,
    bucket=BUCKET,
    ag_version=AG_VERSION,
    py_version=PY_VERSION,
    pipeline_name=PIPELINE_NAME,
)

pipeline.upsert(role_arn=role_arn)
print(f"Pipeline '{PIPELINE_NAME}' created/updated.")

## Start Pipeline Execution (Optional)

Uncomment to start a pipeline execution and wait for it to complete.

In [ ]:
# execution = pipeline.start()
# print(f"Execution started: {execution.describe()['PipelineExecutionArn']}")
# execution.wait()
# print("Pipeline execution complete.")